# 리포트 56 — TX·RX·표적 배치와 β·앙각·원거리장이 유효창을 연다

> ### 한 일
> **조명원과 패시브 수신기를 지상에 고정하고 표적을 그 중점 위 공중에 두어, 이 부의 모든 수치가 성립하는 β·앙각·방위 창을 수치로 고정했다.**

### 결과
1. 헤드라인 거리의 바이스태틱 각은 2.95° ⟨outputs/report13_freespace.json : solve.W1.beta_deg → R90 에서 보간⟩ 로 준모노스태틱이고, σ 는 이등분선 방향의 모노스태틱 값을 쓴다.
2. 상반성 rms 잔차는 β ≤ 45° 에서 2.57 dB ⟨outputs/sbr_defect_fixes.json : d2_reciprocity_drone.rows → β≤45 행 최대⟩, β 60~90° 에서 4.02 dB ⟨outputs/sbr_defect_fixes.json : d2_reciprocity_drone.rows → β>45 행 최대⟩ 다 — 그래서 창을 β ≤ 45° 로 방법 조건에 박는다.
3. σ 격자의 앙각 하한은 -20° ⟨outputs/archive/report13_sigma_grid_pre0803.json : meta.el_deg → 최솟값⟩(격자 행 9 개 ⟨outputs/archive/report13_sigma_grid_pre0803.json : meta.el_deg → 길이⟩) 이고, 그 앙각에 닿는 수평거리는 126 m ⟨outputs/report13_freespace.json : solve.W1.el_look_deg → el=−20° 보간⟩ 다.
4. 장면 방위 72 ⟨outputs/phi_sweep.json : meta.n_phi⟩방위 전수 스윕에서 σ 를 고정한 순수기하의 R90 span 은 0.48% ⟨outputs/phi_sweep.json : verdict.claims[2].range_over_phi.constant_sigma_control.W1.span_pct_of_phi90⟩ 이고, 이 부가 쓰는 φ=90° 는 그 스윕의 보수적인 끝이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 좌표 | `src/freespace_scene.py:72` 의 상수 하나, 기하 함수는 `src/freespace_scene.py:117` |
| β 창 | PEC 이면각 상반성 잔차를 β 행마다 재고, 잔차가 뛰는 자리를 창의 경계로 삼았다 — `benchmark/verify_sbr_defect_fixes.py` |
| σ 조회 | 발표된 solve 가 읽은 σ 격자를 아카이브에서 그대로 인용한다 — 그 신원은 φ 스윕이 기록한 생성시각과 일치로 확정된다 |
| 원거리장 | 파면이 평면으로 보일 만큼 먼 거리를 유효 게이트로 두고, 게이트를 통과한 칸에서만 해를 찾는다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_sigma.py
for D in mini5pro mavic4pro matrice4e phantom4 s1000plus; do PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_range.py --stage all --mode W1,L1,G1 --drone $D; done
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_sbr_defect_fixes.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/phi_sweep.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/report13_freespace.json`, `outputs/sbr_defect_fixes.json`, `outputs/phi_sweep.json`, `outputs/archive/report13_sigma_grid_pre0803.json` |
| 소요 | σ 격자 워커 CPU 20038 s ⟨outputs/report05_derived.json : runtime.sigma_grid_s⟩ · 기종당 565 s ⟨outputs/report05_derived.json : runtime.range_per_drone_s⟩ |

---

## 기하 — TX · RX · 표적을 어디에 두었나

조명원(TX)과 패시브 수신기(RX)를 지상에 고정하고, 표적을 두 점의 중점에서 수평거리 `d` 만큼 떨어진 공중에 둔다. 좌표 상수 `src/freespace_scene.py:72`, 기하 함수 `src/freespace_scene.py:117`.

| 항목 | 값 | 무엇을 정하나 |
|---|---|---|
| 베이스라인 $L$ | 500 m ⟨outputs/report13_freespace.json : solve.W1.L_m⟩ | β(d) 와 직접파 세기 |
| 표적 고도 | 60 m ⟨outputs/report13_freespace.json : solve.W1.alt_m⟩ | 이등분선 앙각 el |
| 장면 방위 $\varphi$ | 90° ⟨outputs/report13_freespace.json : solve.W1.phi_deg⟩ | R1 · R2 의 비 |
| EIRP · 수신이득 · NF | 63 dBm ⟨outputs/report13_freespace.json : meta.link_budget.eirp_dbm⟩ · 10 dBi ⟨outputs/report13_freespace.json : meta.link_budget.rx_gain_dbi⟩ · 5 dB ⟨outputs/report13_freespace.json : meta.link_budget.noise_figure_db⟩ | 선언 예산 — 잡음바닥과 절대 거리 축 |
| CPI | 0.1 s ⟨outputs/report13_freespace.json : solve.W1.T_cpi_s⟩ | 프레임 수 M = CPI·PRF |
| 기준채널 | full_waveform_capture ⟨outputs/report13_freespace.json : meta.link_budget.power_normalization.canonical_reference⟩ | 상관에 쓸 수 있는 에너지 |

## 유효창 — β 와 앙각이 어디까지 열려 있나

헤드라인 거리에서 β = 2.95° ⟨outputs/report13_freespace.json : solve.W1.beta_deg → R90 에서 보간⟩ 로 준모노스태틱이고, σ 는 이등분선 방향의 모노스태틱 값을 쓴다(`src/experiment_freespace_sigma.py:227`). 아래 창이 이 부의 **방법 조건**이다.

| 창 | 성립 범위 | 크기 |
|---|---|---|
| 바이스태틱 각 | β ≤ 45° | 상반성 rms 잔차 β≤45° 2.57 dB ⟨outputs/sbr_defect_fixes.json : d2_reciprocity_drone.rows → β≤45 행 최대⟩ · β 60~90° 4.02 dB ⟨outputs/sbr_defect_fixes.json : d2_reciprocity_drone.rows → β>45 행 최대⟩ |
| σ 격자 앙각 | el ≥ -20° ⟨outputs/archive/report13_sigma_grid_pre0803.json : meta.el_deg → 최솟값⟩ (`d` ≥ 126 m ⟨outputs/report13_freespace.json : solve.W1.el_look_deg → el=−20° 보간⟩) | 격자 앙각 행 9 개 ⟨outputs/archive/report13_sigma_grid_pre0803.json : meta.el_deg → 길이⟩, 헤드라인 거리의 el = -0.26° ⟨outputs/report13_freespace.json : meta.ranges_el_look_deg⟩ |
| β = 45° 지점 | `d` = 602 m ⟨outputs/report13_freespace.json : solve.W1.beta_deg → β=45° 보간⟩ | 그 지점의 SNR = 58 dB ⟨outputs/report13_freespace.json : solve.W1.snr_d_db → d=β45 에서 보간⟩ |
| 장면 방위 φ | 72 ⟨outputs/phi_sweep.json : meta.n_phi⟩방위 전수 — 5° 간격의 전 원주 | σ 를 고정한 순수기하에서 R90 span 0.48% ⟨outputs/phi_sweep.json : verdict.claims[2].range_over_phi.constant_sigma_control.W1.span_pct_of_phi90⟩ · 자세평균 0.83% ⟨outputs/phi_sweep.json : verdict.claims[2].range_over_phi.aspect_averaged.W1.span_pct_of_phi90⟩ · 세 팔 모두 φ=90° 가 minimum ⟨outputs/phi_sweep.json : verdict.claims[2].range_over_phi.constant_sigma_control.W1.phi90_is⟩ 이라 이 부의 φ 는 보수적인 끝이다 |

같은 스윕이 σ 조회의 앙각도 잰다 — 스윕이 읽은 격자(생성 2026-07-29T05:36:31 ⟨outputs/phi_sweep.json : meta.sigma_file_generated⟩, 앙각 0~−20°)에서 조회의 4.6% ⟨outputs/phi_sweep.json : geometry.rows[18].frac_el_outside_sigma_grid⟩(φ=90°) ~ 22.5% ⟨outputs/phi_sweep.json : geometry.rows[0].frac_el_outside_sigma_grid⟩(φ=0°) 가 경계 행으로 클램프됐다 — 격자 밖 값을 가장자리 값으로 눌러 붙였다는 뜻이다. 근거리 SNR 천장이 그 조회 위에 서므로, 확장된 앙각 격자 위에서 다시 푸는 일을 다음 단계에 건다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 앙각을 확장한 σ 격자 위에서 R90 과 SNR 천장을 다시 푼다 | φ 축에서 4.6% ⟨outputs/phi_sweep.json : geometry.rows[18].frac_el_outside_sigma_grid⟩ ~ 22.5% ⟨outputs/phi_sweep.json : geometry.rows[0].frac_el_outside_sigma_grid⟩ 이던 클램프 조회가 격자 안으로 들어오고, 근거리 SNR 천장이 격자 위에 선다 | `src/experiment_freespace_sigma.py` 의 el 축 → `src/experiment_freespace_range.py --stage solve` |
| β > 45° 의 출사 가시성·대칭화 잔차를 다시 잰다 | 바이스태틱 유효창의 폭이 확정된다 | `benchmark/verify_sbr_defect_fixes.py` → [편 20 «수신 방향 그림자»](20_bistatic-exit.ipynb) |